# ⚡ Google Drive GPU Refiner - Remote Worker Node (Helper)

This lightweight helper notebook provides external GPU compute (NVIDIA Tesla T4 / V100 / A100 / L4) to accelerate video refining for the **Main Google Drive Downloader & GPU Refiner**.

---

### 🌟 How it Works:
1. **Shared Drive Integration**: Mounts the shared `downloader` Google Drive folder.
2. **Zero Configuration**: Automatically discovers the Main coordinator via `data/cluster/master.json` on Google Drive.
3. **Hardware NVENC Acceleration**: Receives 3–5 minute video chunks, transcodes them at maximum GPU speed using NVENC hardware acceleration, and saves the finished segments directly back into Google Drive.
4. **Fault Tolerant**: If this Colab runtime stops or is preempted, the Main coordinator automatically reclaims the chunk and reassigns it to another worker without losing any completed work.


In [ ]:
#@title 🛠️ Step 1: Environment Setup & Google Drive Mount
#@markdown Mounts Google Drive, verifies NVIDIA GPU hardware acceleration, and prepares workspace.

import os
import sys
import shutil
import subprocess
import json
import base64
import zlib

print("=" * 72)
print("🚀 [1/3] Checking Hardware Acceleration (NVIDIA GPU)...")
print("=" * 72)
has_gpu = False
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"✅ Active GPU Detected: {gpu_info}")
    has_gpu = True
except Exception:
    print("⚠️ No NVIDIA GPU detected! Helper will run in CPU fallback mode (slower).")
    print("👉 Recommendation: Click Runtime > Change runtime type > Select T4 GPU.")

print("\n" + "=" * 72)
print("📁 [2/3] Mounting Google Drive...")
print("=" * 72)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully at /content/drive")
except Exception as e:
    print(f"⚠️ Drive mount notice: {e}")

print("\n" + "=" * 72)
print("📦 [3/3] Deploying Helper Modules...")
print("=" * 72)
workspace_dir = "/content/colab-gpu-helper"
os.makedirs(workspace_dir, exist_ok=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

# Decompress and write modular files
bundle_b64 = "eJztfQtb20iW6F+pJrcbu9sYyKNnmllmLwEnYZpAFpP0zhe4+mRbBnVsyaMHtIflv9/zqJLqJdmQnpm991vNNx0s1fPUqfOuU/cb4zSZxtf9xXJjT2zE80WaFSLNLxP5ZxZVf5ZlPKl+FPG8/pLflEU8q7/dZFE4iZPry2SapXNRLBfwQ8ivZ4siTpNw1hMncV70xFE8hv8eJEtZehEWN7N4pIp/gJ+XyWXyTBxFRTQuxCC5jbM0mUdJsSfepun1LBKH6SwcidtcnKTjEAZyPAwOz04OXot9mEsfW+xHv0FveedyYxtmXEDly42uCJOJSNLCLXS4d4nPRhe7nkRTEYxvovGXIM6DSXqXzNJwEmXBJM46Y2gjnoRFhL/2RF5kXbH1ZzFK09neZSLgudzA/32KsngaR7m4i4sbcZTFt5F483E4EOMQmhZxchvOsB2Ajbi7iYqbKBOhgDZhzmm2FHEOP6mMqEcg7tLsS74Ix1Gfe+Eei2wp+8YnntIcjYGKNDMmHufOXLpaE/hkUVFmiXgTzvKo/vJMvC7zgmdSD3YGYKT1V8XMEeEDK5AhOPbF56g/S++irNMVUxgWwgLHhW24g7qqW4l+G0cLwAf6B8C21nj1ke/2xSGuKnULPejgnofZlyjL6+L4PVhoCPVrGifm4Hqw1FiM0EaDvnqLE1PTRswzgc8d2FB3AVcNZm0IypavrAVoB5+vG6sFmtrfyqiM+pMRT8+oAUO53JhHRYhv+7/maeIW8vQqF+4iKyPz43gGqAbbzlkFniCCXxYxVqAaavXRGam7GlVf9oI0L4o+xPWxu+rnym1xjSVq6NTTGIFgHmJJbS3surRmSFUQ+X0lGsZgLJq+yZ7rmywvwmugC9vjmzL5krubjMYoC63aL+37sGqk20DH3JG+0EeKPAtYiZhHkzgUeTmapjMguLnoZOFdDxqZxkk0Mfd5mCw7X/RBY0P04vPlBlS73MCRyaqXG1drj+ylb2SLLP0V+aEEougAPY+A7XzJ3VFF/SiZ5Mh5gLf148UyGSH7qxBSDnjNAbmUtQVRzcLMTLMoT2e3kc1KiXECAzX4Jv99tEzCOU57BswFeHwB0JW8n1mpxhMrIt6DdVugEIHSR34Df4zLIu9xi331YqsIs+uoyLdGyy1grjS2EifRI6TLb8Ismgi1/Hc3MfR5XYZZCFJEhC1P4Y3EYSgqTgefBufiLosLkDLkYGDURWoOOEvTou9OVDJrJcBogByFObPumviFoxz/rbYCfAUwRZ0gwDEFQbfrrIOxa1STxLjqtaj3jlp14pUfshhwbSlIEtoTh2WWAd6QCIJgqHlnByYxT0uAz0SUCS6JPnM5pvHdxD8TWAv41lFjh7bgZx/GlBUKg5UItz3BBreVJIdNIvBww9kN25WgTs/p3Wn5/fJo/cJDwhT6G0hn98oUwZoFyLtJw76DL/UCVHS0CXE7BnqpQkIWEiBWkpQswSpfB4iGsA7OZBp60dDUkpn1BvX5uLySiBgM2GSETfWrdgLszeb8eq0ettp1a8otZXIPbs7XESEnwCJObPlj1Uq2tykXlQutLcUuwjyvkQBY1DAKs/GN8CMqKxeTKFrousWo1KTx+ZJq+Na8wvamNZZ125e3lkK01VU1YQ7HStOJbK2BRms2BlMO++JNnIGOgaDYA+qfMh+kMSHhqXQgOXsU5B3JAh+qBaNbaizPs1jTcjZzZUw5gx634EczE8W4HSZNjTgjC7XjDBfSWbCEzQjxARZx4gInHJPKlyYkLDD7EsgdBAhLebQVJ3mU5HFRU+RHgilJs3lADe7DOPsgInWyzc+Xl/nl5VZw9cNmT2yKTQkuIN9ZvACarmThhl2qWmSRqQYVS05j0vInBr8HMaoBdk9cxralbOjJs1Lq9aN298tqdzMrYVIukTjXuItv85rcp2kH6608YRsb1X/HzaW3+y/dYfbn/+5I/i/C1MOKsIA4H5azgqx2OCNdCHGQkIfYxHe2dVB0XRVirXpSuFTjkwJzY1U/tLkRJYXpbSEIjs5+OT05OzganAdHx+eEGH515jKBuVuS/MnZ4cFJ8MvZ+c/DDweHA3NsRTRfYO8R6Eqy/OuD4UB281ip3+3K2HOqZUY6aRLFHzyKLhtdh0VWjgEOQI0M+fJIcu0Y6dP5wS/WGKkHE1Ck+6ISDC2fD94cnw6O1qykFGat4vuzT8eDoa++1jRWnqe3MYnjdd2LT2vUK26DPMrsqmcX76pVb6udovmWah4dXBysN83Kcvh+cHFA1f4yPDt16sn2eq5pTa7YYHETzaMM9iYZw8Wn9+ZiMVo0LJmFM3LNgpibl1VpruvWxsVTyuTF+cHwXTC8OHh7fPp2zQaKLMxvDJWUCzUB1tNEBVn+1gpfs3E/lC8G7z/Irq3eYL3Pjz8p2JhrLJ0YMXKJUYm68dsPH8UhW9fEd6DdkU1syDM116wFZC4i6bA6fPfx9OdhG8i1T0QJaBQatGQL60Faq3148nF44d8sOnR1g62qgk0Ozr19aq32dEOlVvv9Af3z5vjEIXlWbcMYWk+4eeAucujDN6u3TMLpx56KWaBlQt6W7GnZaKdQ7pA8f2VGPieJKsHRx/ODi+Oz02A4OITenr/c6e9UQgCIqGIOejFa3xbQwpdoOc1QlMmja/THiU4KssMciM4onIUJaGTpVLB8UKQzoEn46jvQUu50B9jx9pnC1OBkgFzp4vj94OzjhRzDix05hmfiVdX9LAIlRkzk6EVnkUW3MIIcugPmKcIRyITsJMyxFO6ol1tzNGcjioK8Gyb5OJ2QGMtLFbwbHJxfvB4cXFj9776C7p+J96BSCl4lmNZ0BjyJBM9JNItAlyXRTtyA/F6MorAgeXj3Fezd9wf/GRwcXiBZUAv1l7PXQ2j4ZQ1Xad9Uhoe0BADHiwg72RMwt61ROZ2i/bFcoCXxpfg1HaExMqrs8yKfpTjXaI6DwnEC2fgwOPx4csB9E3xht5+cyHm95HmxApnjUu18i1bla5BncjTVz2AdI5gHzFOCLc4FjOk6HoEcQPb8RTQuZyEqkSABAs+ZRTOQJOEl4xSi3qcoG6WwJCfpNRG278RRNCqvxWlaxFOQq7BkLi7g6wwkxqPB649vg5Ozt8M98qHCOFHQidjpi8ZBkB8HpwevT4DKVmVZcNlFQ+A3KFHtEOIP/wrQfo8FaPOQc7aNFuVLAOQcpPfret8A4UcrKbmh8z1TBlL+5soEOk4XS8UC/uPj4CMM8XVbh7XvrNr0TdVc0mNUxrEeSwtfT4Szu3AJuALjntEAh8MjMfyPkxjwFGVERCFYR9gOaIUx5kQ7EstsQU2a1DSMZyD/5X8SJNXcxdAqtmxO0yfranMxJ2dIuFopu0n2FpQLduyY0nUS3Zke99M0iQzPge4zWAAMYXcgSkv7KLDW8DoivSVHeIQCWoSSkxjowm2EAq/Pp2D42a9nKRA6YbNgKVvBH7pU6Iqu9TsWSevflZxp9lOtviHC9EQldPSEwc5d7g/vamZhNO7hW3VhjQVBF+a26lnL1hP6AlZabyHkivlDD+RHV+1TjThal60OqRa4+KNVEqr0VLVEr/wU1USv/wT1RK/+eBWFoPt4NQWrfYWqQr22CMv4/cliL1b+GtGX6j9egNWrPU2I1Vt4uiBLsDN36SP4Xr0yj+Zh2m73eDBbKT0WoDgctotVlZ5MXOsmVlLZnkZaH0NAe77V5n6vWo2rAM15+CUCcpV3Jj12ZQTpl330tGtmwzVtcnlUWNwRIOzlmtJCkOTA0IEej0HQQ4/upNY1eSyXib0UX7kM/9wl6HkNYD1hmD6qn5U5oyccA4UqVA/O1oqrN/qYGlXBy0QhhokUayBECzIoRHiGOt2Y/PLjJQj283BxA+J7DjL3L6y4nIDQV0ilRC2pGh+qWvzl5OyXAHQHXFW3zA7qKzuAGpOY4zA+np4cvz++GByh9jKWUQGg422BdqBJUbJXCe66vQb9iKQ/WRV4nZwKlK+iS/vVBDu+6QDYYBABDSLwtoQSI06GJEdRJigRo4I81oBIWg5OBvRZkJuzePxlFulzgsIgLeawIGuN0py+lNwv4nmUlgWpk0MC2i9hMb6ZpNfivKTgkmpygJvntoL6E+jHWpF69S7enQ+G785OjoLXHxDUuzs74nv47/OXfVpF/P3z6+3cX9lUgp9TJwageSzng4tz2OxoLBDk3o2LGOTJH+AnSHDSh/QCRGxQKEVYoHW5yJXGsjWP5uj6vQjzL+IiC6XKAUAH8M22MMpYzFBen6fQbpqRG7uAwirKjf4OUFcxgA562Rc0wOPnYEyxKEgUd7jftyzySmUede4sneUavvGXIBwX7CrnCCiW+Nb62DggVmnyEKH094gs9mS8x//UuowdBDWU5XOhKhCMxlma51sLUMDh1xwmMgceHY/iWVwsCZnMkJBwGhWmAqMkc+zc42+5jSdRqsp6vGCXl9vf//ve5ca//fm/rjZ7NNYelev6q1xe5j9QOVEVVE4zw+WCXz7vAcpd4QhhwgServgzoqFA3ZGKKGiOlkgAA7YUszmmQ64QlFgC2Jjk32mJzZauFmrGDiy7wFbF61jR2/cpOmBBeGZNmkLWjQqgaKb0fabM71Flka9DFaYY0iD5VIFmnEWUzYHEJBgrlsGmwF7ighuZLfvi4ibORWXl0sf2txK2FmJxXs6JMzSHlblQsbUx6Sd2C3occloAYDM387DUJnGnTGKQJpW7daps//d48KCP/3nZ6fZvot8+7/3x6iG4V0NG+wztIc+gHxQA8CFYAoLkhS3QegapDUYb4jNaf7T/mYue4qLzglM4IEnUdTU+INHHVfWMsqeNzOjrI7Ck5AvjGDc+ifMvRliZtW5aQ67IyWjV8XbmBF3b4oYIQTbUZQ6gxUUHFukzYSFvnitgXBm6e/ZAEilnHI7H21PQYhLZundB8LAn7qMHI2zblZrd6bajqTt1T3m3uDf+fC1R3GrBiHkdz6IwKRfcK8WrBUCPwiWxABD6qtd71QmZz1DgSsopXQ/BGo7DJHejSkXH7+zuyhBs6FSAwoY7n1RGI4B1PAaanxRkNFMxrGgpiGAZ66DCOkhWxHlectw4mQ6vb1LYXuN0AXxfkjeKogXeoxE2aHEuyhx5vYd4++lXDSE0Q5Pspo7veHRNZTzCv+uQN60NCnRfGQDnIY2eIDjTUmUibms43ErJ3mz5GZqhQZJJcm0pJ8pEjdBQdhZ9TRnNFqrmvq5dA2tGqQEKd7LLjf/Dyvz3l5eAH/3v/xdyaShx/Pb07HxweDAcdHuNVSsLD9ZGtGqrf1XPCC3syNfyKLtlX470OcgNs6cKHk9rK9UkBYji2tyEsK7KBIFrahiasPd8XEbEjkdI92uejUjgWy+kiwG09RiTRzOyyNacAF/c/m4nVXBaQx+yH6sP2VjXOCOBUbEgMNYf/yx2/KSx4tT1FJsYtHokO0N3x5TWnzvpKeB5qtTs4pDX9Uqc09pM5KG2ahFpdTRqBoz1Xrb7oIWkV2uFK/540+OKJcNK3kXz9aYtW3NnLWtHnTWvHn/+h68fdtOrQfqUNTRm37yQWKzm9esLGlWvUsjgkwi1wkhbnQhfLU3IeDCsGU1YA8QXK6IVOTjOE2/eGD7qBJkTYKt2bFxyA8xxRFASx2MSbFcIgi+w6sX4psOj8OCFd6nQ4YAnYYht0JKEWRFPQZFFeGFTDw7K4uPVr6qZueUVtH/YF7ue1kAl/lIxWFXWRG/P8A9lwXIh7mWlB8UBR9u1HOPRyPiojXKTnpcJT14yGDwZMU8nJRSXR45hWFmZEIbF0qfqc2+ukOigO2B/G7P0+jrKGs9Xg6bgP1ON1lvjRfupaussda8SIqWFiSpvoRgGmsR15f78Tlk/LpMA3rt2i3NluICFH5XXAWhao1k0qZzzSGGQOw/4PUqJKiSU9liY4wuoib6ZHgZTAiMGUoDe/gnqUdByFo0B+7B/2Lg4kc84D5R+aS5XV9LWE7Bt8m3w7nh4cXb+V44NkSOvRHu//FxZXtBWjuPpGDMxxWueDIoToG/Rn7cyiIGqIgQplCEkAwx5lRmDcp972IQcf6KTChXENcw3C8PgcXRqrNpeq2MiPnvjIa4osnQXD1FO7UbJgEJBEq3aFiMW5wdwqBfmDKj7g858s8RnTR8Grg3GbtPS+K005yQL54xyMCv/mmCICo/BWAupktmjrJFC4VDHg0MmdqhPoM/keLIqRNtNDmiLdm8SMIlzIPojSotO1L/uI5UxTsF5McVA5FWYYhSGFVjITAmUrgANgmERIAXJi3C+sA9XSngoKtOv/kjSuw5Z5ab4s3O58e27vW/f7307NJIhAOnLAjQ5pdA/Ril1TBj9Ei/o0O4SYJMgIRGXl3jqa4wGVyZfIJIDf8MYlJmgYCqipwBUuavOhkJZNvOW7AYB0rQAVVUcLPQCKP8DmxjF9+KnV/SDXq/vR6E5au0W0W8Fmw8tqywptVSGYkhAuUtLIMBhWaQAfRl7kkUwjzHCoEyiZJxOiKJ8TGKMPwMZJszodN4sug7HSyGB2jZl4Bl97qpfj6/r/T6dlflNx5x5R3Y9SOi/WZYCnT0b0h+rzoSMAc9AOAXRJOvU3aDkSzMj/3uPDRasYZfFdOuPOqGhASJocdDIauCfPlWOgMyNQbLCceT7GGCBcMPDl/1JpL5b4oYDiqrp5oIGTB5JombRbTQL8uV8BFSZfviM9rNbZI30uV8uFvVZELRm4zekzcenb84uN3xm98vy+e6Lny7LabQzVaCLZnpdorxNlSd/fDGB/052Jk5lEGdgk/xycH7KEXP41/Hp24bjFjCOH8OdtnEMzs/Pzhsn8YeXY3+14cfDw8Fw2Fxx55VFt+n1zvPn+Fruz2geE9XWlqEnivBa/jVH3/Z1JH8BFRkDoUSxshJdyDzns6sdovlB8RPsBoMr+NPBAulZLnUH2PJx5cLiCM2eRg5Q0CJ+EBOL4fPkNAb6LHe6x9rVSPOVGb+eDbVJEDAwozKNWWKbpdEa1iV8CrQPeZiHac9aQqF7s6nLjao0LKuojtnX32mQ+I3+cD7DylHF8Nr5JJcSP8s/3dooKWN1ZGHEuLQhP2jcQpM1+yEtZsejyqHLSS+K6rctfVqwNFpepIvOjmGqgHkFHH6KWs09/Hy4YuEM/lQSmU4neQ0KkgG5Sv5wJe5dAtR9EPeydfhLAuiBN4pq7BmpTxmwmetUMWPiudLWRqiEH5LojtCW3Ewa2HwM38+D63Ez400smw2Ko6a8bsLRn1tFt3M4556M5rqrzR+0vdIFrrFRFSliiIRR8bL9in910SoxbTpp2m+bul5yjYQuOrdJMER62fETNZNyvYuvb7ZgGdMMfb+JFlst5qiOFmgv78jAYFruCHNdXIfZZIbh3ulUytGsmpniaU1rFdsiSluNx6CvCupStC7HGGW01hyGXHbN0TcPsWIva4+S1Y51xngEwk+MStQkDq+TNAchjzZMR8E0RRE2VtCstZHm4UqS3TZYZlVqtHdhlqw1WGUno91sgnVtYCphYU1IkvS21uBI3vyqoUnpY+2xgeCqU+kOxfvsocOeY2NIivPZH7xqKLfE1FIlKcJjE9V5yQk0YorxjXxdSjnoHjL4zuctHuNV0xxI1PVPxI4hUeOuqdS8nBUxsQEMwgA8SaeeWa03g9rN5R2/I8L4T4hX0t4pS0+JcUYEGk6zCQx8GaE9r66Kc7DTXtm5jWw2Y/Lj6PMmvNm8UkxZ/fZx5qpHJUBULSgJCNuxWDV8pt+bVya/hvcScaGSwS9qyRcZCZufqduu7lLOGJXNrUXMPtdkU4TlTZw7BxiajQu6NEP9KHvmTTQDhUbGOT3erEl+EW9GSSuJZJnNZvGon0V/KyMMSDVfE5mxXy5g0lpSynK0yFJkKGuaUD3GU85ReVmPWFnD5E827up783pRBnEyTdneYtIRY42OEzw8BfI2x4qIG+DEd5g/ChH39NPx0fEBHRDFUuYexOZNAfxyA7ulkChYDlh/qMcBn0Ad9WK3WTgP5iMstWN8IBt2xrVVPw8+2wOGKe5rsO1z2gV4vSiLjrlJPl9uJLcxMMqtfB6zwrm1BcuZLbdgvPs43h4jaJ+iA3s8ikDGU6oaTLD2x/ltL0lvIoxlhj/KBANZN64sVQDp4b4ZumHFmOGzwGRSSDAW6hv7XygnUVn088UsxoNmPczj5GgGVBtUgn3xwraZwtJ81lfjiuxyUPzzjp08kYtWK4Il0enBpXevbNchF1frVLf7/GpNs5ZGU7AxzeiaAA8eB+EtCDYoqjSYXyn7VI4wOP00OD0UN9HtePvm+Y8vWVzGw4mIvjLlGklCsOFAAJKkotme1Y5TMOvpdL6IriU+qN5w7evl7qHxDajCvtbM0eDT6ceTEx9NxcHzvDnNIA6B7FU4IevDSvj6wn0w51IwLXPWLDqad6LxvNpFFiLmKxeW5jnFuFS0GNTWXrQsEseuYkrwO6ideBrTn1ipZQGq7aDh6LjUz0oqXatWs+qSnBIPy1MSNvj3G7civLb9lNSr4qH43XIyeAdglqs2LcZFAugmcm82B47xabNHZvXUIncWHn1yzTSd9TZcwwQ9nmHM3DtitkzKPYaqgxIUo3SOp4WRVZxTCHKmYv0TNC/Xjsw8nuBBXk6VzFxcqJSR0gZ1JM8+5uJ9CEC9xcyXWiaiHkgcYTxXh5Cf6/H+FCrMp4SV6Yc3ak7NVNyNaAdbwjIpkGIUEfqUJ+oYeW5kTdRH4LGVoVXm8Oz08OP5+eD0Qh7MwGB0ZXQgYyHwZFAbgk4ezaY9SZGCeOL3FcI8UxA08XRsmgUgXDQVY88v59YKOIK7LXYPHxxAXwkJbEzXZAatHIba4RekjHodk730pZEceBXLDEwk0WZef7mI8llYf1Acj+jdKSyRLtxSXxV8oPP6byg/VafVgvtqgCrb0UNDoK7TOuKlFuTLOC469+Y0N9UsQS7uOm3wiTf/2liLp+bbz/iPyw1KGImxB2ZBJl/+yqwHsCPZGIjcX4Y/3CrCBwUCOp+/xyIhMPpayLS1TZzD/c6exLFd/uOhpdHm0wZGlSkZLgLeowFMdIZ8RY6IxN7pLA0L6v7BtB6ec4YjIFwgGI3jQmZMklnlMCQlS5ECaH7fKmJT2x6cL8F+q2Bsp4lUONJYweIXKgWwFeJottdzPxgBPs0xoT3ZQvfRTZip2ap2zGYs8ZCOd1D65WpaHrbiMLZxU+avZuZGsGOHvv+kve1z0x8tmMd+1uSK+JjR088EGTMBm9Py+gZEQHF+8H4LI99V/ug9SkCwPYlut/ObOSU5VsIrodhNmKN0/mpnR7x/DdJUBGIpnrrQyetcuc1huWRDDpWhCOUAJXTqfd9OJu/Lr8ftdj0Jq6tva1m8YaokFXOoHvYflGgmqJvxogNW69OMJQD4vJf8p2ENaLJ8LACPZ1lZhmWGvnllI9dMAN54sWb4uTnd8dGSQTxhZModzTiM47OTMhHns5KsdVdNZA30/dcMzgOtf81AdJ+MZwSuH6aWx1SSjaDm49QEKUYGP9dmqkt9+Ax+uwkxs+1tRJE4mDhT5lwn6VXj5JuYar46Ii9GUTjGOFOKM9E62O0rQug7gs/SAX1uzM9Ut4XpmpsTZPe3MUx2W+YO2NYGx4QDtBhKp9RvKVj3hYnrSao3j/tpOesbsh9+v07rmJVU5U53WmpIE71ey6/6KtGpTGtqt64zzvWafIO2/1zPZeNcSmJetYJaMx+qYy15sj2P8xwTvlMwqcqa3W9Gw/rqgQXnCCIPgpRFP1+ZghReOUI4JBNwc/4ZU6HFfSqTg/pQbKPevN7PdqrQhRayo3Y//pBhOo6mvPBQOmuOSmlfdM3ZAdJzQqQsqpJCGdNraKcps6UUpzjliJFsw816oee6f1o3/T63u3Z3+tRhD/6coJnGRGG+rIC8F3VyVBlRrMHcpc4q1hj7NgJV1FNHHTfIuy2S7BqFLZl1jRorMsarxyPnTvhQV6LPyS/s+o5rNCWMJ0RozOPeLA97D2xxL/K2o1xLru5vY81kyOoBUu251IZz2rflqZZQscRMbKwtATABpn2HsMiHDa2/Ib7SLqZvpqNoa1Kipom2THk2jKydGeb41WT0KCHzSKHrgvLgrqkOWo7B8cJQrSTlNoeHiZo9aabGtgVQ5nRWN0/goLyyYpT0w8mkg2U9sHJGrZZEljfHL5kHdOfUs/pG3X1cZjlLSSPPXV18R0B1Jq4yNuvL69pthXvJkN9gy0O1ZvxUk3HVfbPZuHkkbtk1zcdyldcwIePTrmmvMCXj82jFudHYbVzN8TiTdx0jxbUp49kTYqPwoQOp++T67aNdoTNtmDka4fapuEz0WKsHjYomTFfZ7kDm6sDfXfOymJuiWJgBrvZDgecBd161YVgLm+s+E2dV2sX4OsFgut3nf+jvwP92xTaf2qcj0WFOp5+BoMl8X0h+kGZkoI3c6NqKyvnQSvPXkRgIKDCDakDqQi81XbIAV0O0vq5kIY28tYaNjOjCw2JGZkVeV6mIfSOGy2R8k6VJ/PeoOlviSQfne/joaezb76t+y23h41waoKsOdIOOcfLWvSrRftpJggbQZiNc1VV7O4+gHeoxjW/2o5LgK6z4HbxfWrPqZBa+QrsA7ogyGUdBkfKxOumwQatWmasErpcb8WQWmXtaBrxx7PU4nQGRhbFskSVM5vmRaj9FZHscTLry3LeVPEnCnKDnyjuCoROm78SJUa78HlXZ6o1TVo/naHH/OPW0AA+7Wh1p4NRi8FIl+suN3Yb9GlS5ja1Aa6d0Fl3jaSgAauAUrstqHg11R5Wjx7hUzpdTbaNnSs6KhPiyNJoJF+2xN1nJHtFrS45wq++6a0s6JXc+kGIJlkcHSBde65td444Cn22tg+vqfj4DqR/8B9upvUpi4JZptk+QGEhMmJTzBV3rCWPpYa45UIP2nzcQv6l7pGc1wDTATXPgPx1oBXNrpZ0mhvBkh8aalXVhjsL7KAoG2E6QYQR0FgWKPGbtZNEK38HHpmfVbcTFTVjoYTqbeUWETYrJLA8NYl56WbV8PMW0VXdhLhOjT+T5x9HSkHGIobA1TiZSSygbu5FgZGsMwlHBibDiOV3FWaAaw9EDW4rOyDuV8USh1kWLye5/6E1tfMoDNHjiSjjurX8eLVor9Ab/0xXmDXJtht7fSVRRz+9NL/3mLO6lSaA0VsvvTMPH45V9JP2RA6z7s8rLrAbygMrlxrvByYcB4ejUOIL4UhzY1ARJg5oE/GT6gVYK3Ov4crMNlJv9ft8BJ0dEuLIjkcd9/se12tiBJD6jDUZ4SCoTIJGRDiub1XhThtlhgfjOzQTGUGHvl0Owz6t8WTqB1smym+2CKHRZoP7giLH/Q/i+mrj9A8jAo0jAWoEcvowtstmvlm3qbHreFp8g6ARooglkpL8UbaJkQnc2yGNEKAruNUZMqTg7w3vsP8ODjzxtRgd5w0S8u7j4ID6cDS/QHvJ2cCHkSFiqGGXpXQ7UiwPP+aqI2zTGa3TScjKdYVDjCLMcyvsynD0n6Xw7tTH0YfWSLVKAVPdW5Yd7BR4jus5FDZhJIFVXs2l81IwctZZh9BFmvXVwTdfC4bGA9+nf49ks3MZbYzq/xJgsNxenF2J3p7/zJwEvfnz5J/Hbjy+7eEh6Fv0SjX6Oi+1XL/7Qf/Gj6Pz87uL9SQ9Y85dIvAW5Nu2KwxugXdE2ZRbG/4lhOA2zWFZpcDodjBGveEThgn0EsN7bvL+8Vf5ziwNit46PXC3drPDgMAmCXszJ5rSI7QYgV7pLTspLVyU2qDUft7pcBvT2sj9t62K5iFQGF3eOzgpjsnfjsEz/XG4leM1bZ1+Nsae625f/2oZ5RHmrNfhJah387pEqn5bF/oudLps0c5+/OKY73vFwBbJePJn+fMeXvgyfUTpZUpJgKI/xi50q3UMb1HjutGkqu3LewbbcxA76maE+7nY+emjlGZPjRjYSxvMAM0dfbiivTkWO3JFIUYgOZVqCEFGWe8yTOIkekDHX+xYzbuF0Yexu0i0vNfBN5uP5ydPnQmoUH52f0DmHqgQawaPqGtPHzRmQOAE5vEoN+jtNW6ZIp9muQztXZZWrIPQV0+dTw9b86x6dia854wbbqFcOrYKmvPr+ea0gswm0MnvSPq+C/AlPDz4cE0awFPmaZEyHmcEggkW4xN32/4051GuSXE/BGUe7QgGZriKvwKrbOjoO/+7aWkxGbmoqZkpDlxvb4SKuop4UEnBm1WotzOwVGaVeQhNJLv1oysDb5SQhtYHWSMGCD4hm8TWnSqyrq5cSzj1ruSyklqBTJ/8t2BkuoXogDaCDXbt5r7p/2PzGSiaBzzPDfWSGwMksd7VpgNQlT3ggcp80yS0JaQ4Fjfh1HSjae2dfU8WJun24rmPdxuLosTARvKyCarkC29MChWRNey4N3LgOKGoJhF8RLXTlb9oI9WkK8tFGbPrbW+N81LOeo82IznGCfNZoQA7QuPVbsQvcYTas7TJrdoEPBnY8MTLIM2LrsAC0/Zix4GOgKDbwuOot5wXU87u7MNHUU416RWsN41M6XD15Srat7XI33F/72gRkA5haeddU1Tb+ZqRv9SjXbTbg0JoscLyrE2AlP2g3XfAhoYq+gyRUd+1POPskO60nLr5JTL3c4NuVWfBhJiQvkpXpCv8kQU7ikclT2PsxUvKRwY/aDZGVg8aR9dhwWMt6OYyi9rdKk8gcb9MZ5+uZQbyZB/DRjR90s0514yudnKnury1SWi1HAqylPzketJrcPzgFPhvS4JWSbjyK97qyj+Z/7qleunbvdLeH/GgJPz3hWQDqttXTpozIpozl70LKV3UuRD9CehWHekf99EK8U3Pd43vBxbN7vcdNOsQHUPxts/sA8qXxbTEuNntip7/T3evvTh++7Yr/LcwS00WOJaAq/OXDxywyPZGVFuk5pbHKzvYhnc3kgWJkvAmmfayPa/G9wizh4dlfB9+S9A6PNvoiCOQZSHWS0WPWRtvZrw9792OgM9R759eeGFOKsAUHazaei+zHRTTPO12m/Xfi37CO15KsagR4A/KcLXbfAQmblSAfc6v795u9TWbe9qDxiho6wWZPxpN5ptLcqQ9jc9RL9O/VFtu3rdDfVSL7/r2eM6WPVwhxrI0m1T/cmzN7MBXBhk2rxrhKJbkJc2mXsLnjr3zwuC76azoiIoIoaxYd4xYwylYbA4tv7VrlnwnKYkFp92qXikx6BPoIowJ/i36DJWH8tDpVUCmTIp4pKDRgEY6qgzPq8WC7vDMdDq8hOKCa2UWzZ4puQeeNqffhkRGw+XwWRYvOc6d7Xk+PJQKfdRXhV+IQURDvl5cESy2KRqxwBcUPYvdhu/5KqV8k3JCapVPxl3Sk1+f1J0q3qdWLi1kELW5Cafi22W2yrmTR6sAzlYOgJnI9wd1SWq2eqOYgf6dlwWOus3r0OD8HXtwQzEd7fJoarxzo7zSbaIblCG9klHRQDgMFEQT4OFqDDX+VEUZtrT05W6eAtp/2aiA4xUxwYFnzjVNBBxUWN377DDLrSgnGSuqSgmPbWcGFo5fyinpAaccukU7lij27r4DyoC9fh3GYofrQta09hISU2r3NWMOHwEJyeUQTlmDCGRrJlyDNJ2jy6pomQo0grETiZqSU886NKQMmyvZFyBIBXpVJQBiFQFPxJJ3M9S3N/nwrR2HlHPvXY+9X4ZcO46eg1/PdUKZ2FufcVE0xqxESjawpocQiG85PQyppAeSuNf2FzuWqCzB5JJQP3+uPZY95wCd+zbxGXow6Uhemiiy8k1hjBuJ6LXO1dRpaj0I66i/VMFDP0C9Xg+LpXlcr7KnR7arOBGxuA7/xu2Abfar//fyppi/1++3vPSXWdJ0a22iVP7LZAekaMtZ2R8KkH+GP/KbNH+lHCmNE2rkYbQtQ1MlIRrci+2uKcLWuX0pHv+KGXbBQYZ+NkcPxHunROvffH+Up9md1BRI+63jJ2sgZbU3ezXKDKgrCiWBBNSV3R6v3ywmQYqc50yBJfpDa7llGjR5ezFwo7oaC1jqRr4PfonGJMaUqJdUW8tdZlIUYq8qp7YosTHK6nQHvZaATzcMhXm+9WFacX9sruWR7iyiLQe8f11aUvMeXu/J9iEDxkji/gW768y+3xhkEO4rW8buFdzJejKG9TxAB4QA/6IKXppciNnnr2MLalb59p8F4ei3LSt5BVxtjJg+Z/RLW/v5BW0tmT3X7ii3rOnLF2apSGm82Dsnze+luce54tQBhsF1EcmQvduoSX24KQN44ofRJ9xKLHoL7umsTWbkypylcq2mE8Iq269ZRTJrS1QzVFHp1ly257NQlqI9OZqduQ/09z+zu9sWbqBjfEIOXidz2xEW2NM2ndO+x6BAZnYA6Ko+f6jkZQEWehrOZEnYmmqmWiYwOujKZ1HcZOxHM1glfNBBZCHSlz+Fglqccds90nre+90QaxQbFeOMD3fqDwovpUaQrOetrfGrMnj4VtX2uIE/cJiXA0+8ybgzbNCbU43NA1/IKF7k/OXYTURo3tRJEsYA2GevuhHqwMp6i/Zx1w1H0uhlns8hL5MmNGCbXUeelkzsMTxjgCFb2jo+Vq1P2aZdyNx4V9PJd/uK/shEfl5w3u9IcHH9U3DeM2mzAN3+3mmYv2u2bN1msarEhnZV95aQJgV5NvlfEk/pEFHxaQpI+D4EKi4oWX0lSROxcWvw6IJ90/0R0hy4llrRHIzq+uHPv8hjijAQYX9piFPTfu15DAQvYaKV9BS3ORq4VlrrnL6YvmjRNpbjXulVfDGnadeKilTqaD0LcT7TwWazx07YiKduaHcWOcZwAI/yilHOfiqp6acYiuQbU0lcij34rkhimZTaOXFsQgxW7XEYF3qod5XlMdwomKkw/Y6BJK4COdXwVbgO+tRibP3ekGaQWt7pXpvMExoXq4VoGZaetRquxR1eaBnP9emMPBm+LjpaZzh9D1IjLf9h51YTLnXvsm1xf4v3rLhn3AeDsynWFiRqp7eAiymf0uoxnE/HmDSag1vSCcTqfGxQ7h3lFiI14rn1fydHK+KJ9ZOaalDMzpcE4mzrV4F0AYhHWeKGffbwNcAxjpzzl4OVv3Mu7/vMfXwGg3w0+HZo3H+cBZe5GUxD8yyf/VbOUFeAAa2hvNRNHKDOGY8JyT/Jw9wgRduVKsVU7nv14G4DiKsOQtsZ7tzwdPTc3pQKn1DAF/1i8lC/Hf+PyGMUJEOxa0UgNyQb9Pc7i0W8EH245m+oNO0O4jbLlNMREClqfnv6eNnktY/k/c/KvnjJ5Da/nE27XyOGOF+cA+5C/lgbAngkZnPHp/OB9nTiaYsKRSuI2Pv10NDjcw1vl8P5I1ObxyoOqyCgtilmUgBxvoGIT0GGMwH8LlDoRADd3ZBGQUnA5CdnM3lCaLjaoyNqVKQsbVOGb/Wrjtw/gliGt1W3s32jme15Hx4q3FbpHI/AtHmMJkuiaUrUFRa7SOn2Jgr9HWeqro4JFJNrFi2hv1ylX6a716yuDLEzKLMiJgJGPrKPZGuQnbP/5S/ScuXnWTsIyAT1T0mSZ7F8qbMhZ6xiWeZrERaqFU61iKz+GuyD34K3YiU7t02kTs0FFw7TRdzbvq+lIHyU6JykavDnDteiYYoYTIIzTNC9J+EBWSMCGnrx3VL/64MPxh0HLlQjGxQmjcoq8eX/XADWnfBjRJYBNARiY/D1YjKWT03o/XeTe9/kiovBivNrj0rilztUc6A5dvKAIGBFOQV7h6ZPgsNQ+/eNPD40PaqdcQMsPBOPcb84PtOK4XT1RxmRuXt4egs1+3r1S42kJ1ZQmMYrGoRTmsl1/ja9KREANzLyQoJVpgYWxfG0zbQAlT5Ia0KbJv9cdI1rWEA2D8umrhieS6K4V3xxaVomHS3QLmtgWu8HOzg7+3ydQG5V4jwCv6vz0U3+nJ+bhb50d/KNTtblNL3fxpaSB3S7JxpIINrUvoQo9aDDFX78Tsmk11lw5fL4aRfFpCf3SH8CTDhbdMmgW3c+DvrSWrqGo5va1gi49hyL1Z6X32a2wViyFW42Wdq9a2JXlaWH3qmVdWV4u6562qKvrVBmD9GjH5loPLRhMSkKcQJPJOOrwmvTo9jw2rvEbdRRlhNdOcFRWa1I3fNaLXYomu8LR8hv4PXWP4b9mNpPOXZzgzSvVVSbdvniTRREFQ0CbTcq8/hB/kzeiF43Irp71TjpQm3ch0DflKPVHf+nPU+LvsZsv8Wy2atSPc3X4nvXmjc+ajpKmp91NYj9PABo+7QcX8PFmlrCfFREnKNWui+BKwN8T94rgcBQxBhErkrLX35lS7LD4LzFEeqFKE/Hwx/RXYzXlSiDcLbyfg1XUmPaBQDu6k3o0PrvbzJFtbukpWu8Z9yRxqxW6cRfI5eGrUVeZF+suQAxA+tGwUIrq+l3r/wAr4eMthD7/Pj58X8y4z99JwwLdeKfVJI7kodkkTl+bTeJeyPMF9U3grZU/6SF4pP4mPsPw8b7ISQQrZE0Xbxn9f2nB1PXhakSUY4bPcWoW0JYoaLwP3dN7lYJe7+olpshfLLVrwdi6rodL6C7lum7bJeFWMETLLeG2p6rCsJ4dUWE6JmXwqsfobVdrNn23H51tt3sbxmmCHJrCVTRyZQ5/rB1CDEM0ntcxKraR/FWf42SXjmfIwg0zxtrBBge8vQqmVo8/9gkf9dAcdAinWYgnfPHeBm1ZHiVx+KWLR0gSzVLDExJwNRwOk3fvUoBJMEtha8lsgFo0lCc1Kh914DvC2I0u0OsE24iumcVl74AKKp7T3xhZmeGXrhtaOZnBGsZ4Gei+HkzGSb2NW9Ecu/fqmExahtr7/MKX3rvKPGytXphXbkLtyJLnsAVGk3nWuWFi1QTRxOi/fa0p0s8u/Fku0pWMg3KrrTpCWGtbPucgEw4FBCOQDrurkGRf/tuUx/P3m6V+0uLrukHzfZgsA5kvch+Uw2Un15PzEFrn1cEuveH+bTgro9xrhZN4abbfOtc1Tng2diPXqMlypaG+V1VruSxLq7vTf2XTH389Ded/AHHYO2p9X0AZFMO0V9+C6OGTvdTTppw40u8xtLsnfgHpG0VfolLVUUH9itEmhdqiHDXRBIrUkGOPuJY8CBZRcGhMUZ/pAnQV9DTgSIy7TpFeygqMWzaFbLY7kJkR+P6Pf3S5/iozxR931DVP0L+8NROv72SPBSlh5gk+O4ixpRf4LhnE8ZFqpz43+Kh2YHR77SKG/VGmYsFv71+T26/7qB45aPed8hQeqHhe5LLifvOvUb4pPdHuTdd0ynLzNBUd9CC+kXGH3c3HzflQIcd4qebuvZf2QaIvcN55OSvCJErLHERayWsf1ecRbihxkl7LY0ryniGc8eD04PXJ4Eh0PkXZKM2j7iYH5lGDAAbei2ryR8dDWfw9yk4FEFLM8ztbrg8FA611UY1Q09w9+s6pJl77gEg+8d3ZQhJKFfLXDGNHphP6zagX9JcMjN/n4z62PNUT6Ebd7+CLXhczqIFIl/hy1xbsnLBFDDkJFcpY2DER7RLQanIwhv9Od5Ec1CbIgTobzPeLlVlNELTEQ369synFqSbbAUV+ksxHgcHqXmlBB8KmWZR7Uv18PD95orholagixTyXDPq5/+ouHrcuR/VNFzRNbZ3WWZO11sYu3CIXaIvn5/ESDPoiSz6vvfpW/NjG5jUIefJtyPBDnbE7CCDT2vozjKNuUl/4t8qe3kap6hOeL38SF/FiD1OWh2WRbimEWYJo/gWz7oIwgQf8oF9KLu5DWVS2Uvq6aV0dvSno1HyEoR9Qb1hEC/G8edia5GKLbupp0nueiQ/OyRO62KIIlzKvADlSOGpEkEGIT2cCuxOSLgMXUqoF02azj8eJ67IMqhxEygtOt9sumZNmpovzVw490Nv1dMvB4djLioL4PNLnp/v7wsLKTOC7E7quaPj9qrra2/bq0v9XVaTfnEagtZ70A1b16Pca9Sp/YFVTvunJmI0V1R/hGrTdgg1EzL9URk+sd9nt6WHsHMUudYFAXmqLSZc61sat71PAKTu3j5ufowQFSZaoYDdc58ZN8EoWYQVhQDnHmEsjqkoZnoV6QPoIan4xlAllD4wKKbM53ckJS6FqX7bJsrw9sX3rt2dy+84b2YFpGePukOUrViTNWz9Hy1EKgvhxAmQvKxd6qM5aRBmdsRIuUiGJVVPsdi1zaEMMb8qCOAmZly1u4DlHyuNt0dvT6RQdTg55ljW9Odtb/EO2a0ha9oCEBQTVIGDXVkAYGASVW8vGTehj4+H/Atajq9M="
try:
    raw_json = zlib.decompress(base64.b64decode(bundle_b64.encode('ascii'))).decode('utf-8')
    files_dict = json.loads(raw_json)
    for fname, content in files_dict.items():
        fpath = os.path.join(workspace_dir, fname)
        with open(fpath, "w", encoding="utf-8") as f:
            f.write(content)
    print(f"✅ Deployed {len(files_dict)} helper modules into {workspace_dir}")
except Exception as e:
    print(f"⚠️ Notice deploying bundle: {e}")

print("🎉 Setup complete! Helper workspace ready.")


In [ ]:
#@title 🚀 Step 2: Start Autonomous GPU Worker Loop
#@markdown Launches the helper node. It auto-discovers the Main coordinator and begins encoding video chunks.
coordinator_url = "" #@param {type:"string"}
drive_folder_name = "" #@param {type:"string"}
#@markdown *Optional*: If Google Drive auto-discovery takes time, paste the Cloudflare or Gradio link from the Main coordinator here (e.g. `https://xyz.trycloudflare.com`), or specify the Google Drive folder name (e.g. `colab drive download`). Leave blank for auto-discovery.
enable_debug_logs = True #@param {type:"boolean"}

import time
import logger
logger.set_debug(enable_debug_logs)
import helper_worker

print("=" * 72)
print("🟢 Launching Autonomous GPU Worker Node...")
print(f"   Coordinator URL: {coordinator_url if coordinator_url else 'Auto-Discover (via Google Drive master.json)'}")
if drive_folder_name:
    print(f"   Drive Folder:    {drive_folder_name}")
print(f"   Debug Log Notifications: {'ENABLED (Verbose)' if enable_debug_logs else 'DISABLED (Quiet)'}")
print("   Keep this cell running to provide GPU compute.")
print("=" * 72 + "\n")

try:
    helper_worker.run_helper_main(coordinator_url=coordinator_url, drive_folder_name=drive_folder_name, enable_debug_logs=enable_debug_logs)
except KeyboardInterrupt:
    print("\n🛑 Worker stopped by user.")
